# Function Calling

* zum Ausführen ist ein OpenAI-Account notwendig: https://platform.openai.com/signup
* hier https://platform.openai.com/api-keys den eigenen Key erzeugen und eintragen
* Kosten
  * https://openai.com/pricing
  * https://platform.openai.com/usage
* Installation und Quickstart: https://platform.openai.com/docs/quickstart?context=python  

In [1]:
import sys
IN_COLAB = 'google.colab' in sys.modules # True if running in Google Colab.
IN_COLAB

False

In [2]:
if IN_COLAB:
    !pip install -q openai tiktoken
    !pip install -q umap-learn

# OpenAI Client

In [ ]:
# TODO: hier https://platform.openai.com/api-keys den eigenen Key erzeugen und eintragen
OPENAI_API_KEY = '...'

assert OPENAI_API_KEY != '...', "Bitte tragen Sie Ihren OpenAI API Key ein!"

In [4]:
from openai import OpenAI

client = OpenAI(
  api_key=OPENAI_API_KEY
)
models = client.models.list()

# Function Calling

* https://platform.openai.com/docs/guides/function-calling
* Aktuelle Variante: Responses API mit Function Tools
* Ältere Variante: Chat Completions mit `tools` existiert noch, wird hier aber auskommentiert und ersetzt.

In [5]:
anfrage = "Was ist das durschschnittliche Gehalt aller Mitarbeiter?"

In [6]:
# Falls dieses Modell für den eigenen API-Key nicht freigeschaltet ist,
# kann hier ein anderes verfügbares Modell eingesetzt werden.
model = "gpt-5"
tools = [
    {
        "type": "function",
        "name": "list_employee_ids",
        "description": "Enumerate the integer id of all employees",
        "parameters": {
            "type": "object",
            "properties": {},
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "get_employee_detail",
        "description": "Gets the employee's details",
        "parameters": {
            "type": "object",
            "properties": {
                "id": {
                    "type": "integer",
                    "description": "The id of the employee like returned by list_employee_ids",
                }
            },
            "required": ["id"],
            "additionalProperties": False,
        },
    },
]

def user_input(input_messages, content=None):
    if content:
        input_messages.append({"role": "user", "content": content})

    # Aktualisiert: Responses API.
    response = client.responses.create(
        model=model,
        input=input_messages,
        tools=tools,
    )

    # Die Modellausgabe wird für den nächsten Turn vollständig übernommen.
    input_messages.extend(response.output)
    return response


In [7]:
employees = [
    {
        "name": "Oliver Zeigermann",
        "position": "CTO",
        "salary": 200_000
    },
    {
        "name": "Chi Nhan Nguyen",
        "position": "CEO",
        "salary": 250_000
    }
]

def get_employee_detail(id):
  print(id)
  if int(id) <= len(employees):
    return employees[id -1]
  return None

def list_employee_ids():
  return list(iter(range(1, len(employees) + 1)))

In [8]:
import json

available_functions = {
    "list_employee_ids": list_employee_ids,
    "get_employee_detail": get_employee_detail,
}

def execute_tool_call(tool_call):
    function_name = tool_call.name
    function_to_call = available_functions[function_name]
    function_args = json.loads(tool_call.arguments)

    function_response = function_to_call(**function_args)
    print(f"Calling {function_name} with {function_args}, getting {function_response}")

    # Tool-Ergebnisse werden als function_call_output
    # mit call_id zurückgegeben.
    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(function_response),
    }

def execute_tool_calls(input_messages, response):
    tool_calls = [item for item in response.output if item.type == "function_call"]
    for tool_call in tool_calls:
        input_messages.append(execute_tool_call(tool_call))
    return tool_calls


In [9]:
input_messages = []
response = user_input(input_messages, anfrage)
input_messages

[{'role': 'user',
  'content': 'Was ist das durschschnittliche Gehalt aller Mitarbeiter?'},
 ResponseReasoningItem(id='rs_029ba5f87978f172006a22b24368e8819e8d2ebce290bc6ba7', summary=[], type='reasoning', content=[], encrypted_content=None, status=None),
 ResponseFunctionToolCall(arguments='{}', call_id='call_SvXAJeOhTdW3HkEDobxtBXUw', name='list_employee_ids', type='function_call', id='fc_029ba5f87978f172006a22b245cdf0819ea120e47f5798d77a', namespace=None, status='completed')]

In [10]:
# Function Calls stehen in response.output.
tool_calls = [item for item in response.output if item.type == "function_call"]
assert tool_calls
tool_calls

[ResponseFunctionToolCall(arguments='{}', call_id='call_SvXAJeOhTdW3HkEDobxtBXUw', name='list_employee_ids', type='function_call', id='fc_029ba5f87978f172006a22b245cdf0819ea120e47f5798d77a', namespace=None, status='completed')]

In [11]:
execute_tool_calls(input_messages, response)
input_messages

Calling list_employee_ids with {}, getting [1, 2]


[{'role': 'user',
  'content': 'Was ist das durschschnittliche Gehalt aller Mitarbeiter?'},
 ResponseReasoningItem(id='rs_029ba5f87978f172006a22b24368e8819e8d2ebce290bc6ba7', summary=[], type='reasoning', content=[], encrypted_content=None, status=None),
 ResponseFunctionToolCall(arguments='{}', call_id='call_SvXAJeOhTdW3HkEDobxtBXUw', name='list_employee_ids', type='function_call', id='fc_029ba5f87978f172006a22b245cdf0819ea120e47f5798d77a', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_SvXAJeOhTdW3HkEDobxtBXUw',
  'output': '[1, 2]'}]

In [12]:
response = user_input(input_messages)
response

Response(id='resp_029ba5f87978f172006a22b2465f00819e89242a7f0389773e', created_at=1780658758.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5-2025-08-07', object='response', output=[ResponseFunctionToolCall(arguments='{"id":1}', call_id='call_Os5FTdeLEe9J849KDwqnRftW', name='get_employee_detail', type='function_call', id='fc_029ba5f87978f172006a22b2471a00819eb2394fe2409e7327', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"id":2}', call_id='call_02hxmRWs287bpjV60rYdGQAQ', name='get_employee_detail', type='function_call', id='fc_029ba5f87978f172006a22b2471a10819e83b715d71e8fbe83', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='list_employee_ids', parameters={'type': 'object', 'properties': {}, 'additionalProperties': False, 'required': []}, strict=True, type='function', defer_loading=None, description='Enumerate the integer id of all employees')

In [13]:
tool_calls = [item for item in response.output if item.type == "function_call"]
assert tool_calls
tool_calls

[ResponseFunctionToolCall(arguments='{"id":1}', call_id='call_Os5FTdeLEe9J849KDwqnRftW', name='get_employee_detail', type='function_call', id='fc_029ba5f87978f172006a22b2471a00819eb2394fe2409e7327', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"id":2}', call_id='call_02hxmRWs287bpjV60rYdGQAQ', name='get_employee_detail', type='function_call', id='fc_029ba5f87978f172006a22b2471a10819e83b715d71e8fbe83', namespace=None, status='completed')]

In [14]:
execute_tool_calls(input_messages, response)
input_messages

1
Calling get_employee_detail with {'id': 1}, getting {'name': 'Oliver Zeigermann', 'position': 'CTO', 'salary': 200000}
2
Calling get_employee_detail with {'id': 2}, getting {'name': 'Chi Nhan Nguyen', 'position': 'CEO', 'salary': 250000}


[{'role': 'user',
  'content': 'Was ist das durschschnittliche Gehalt aller Mitarbeiter?'},
 ResponseReasoningItem(id='rs_029ba5f87978f172006a22b24368e8819e8d2ebce290bc6ba7', summary=[], type='reasoning', content=[], encrypted_content=None, status=None),
 ResponseFunctionToolCall(arguments='{}', call_id='call_SvXAJeOhTdW3HkEDobxtBXUw', name='list_employee_ids', type='function_call', id='fc_029ba5f87978f172006a22b245cdf0819ea120e47f5798d77a', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_SvXAJeOhTdW3HkEDobxtBXUw',
  'output': '[1, 2]'},
 ResponseFunctionToolCall(arguments='{"id":1}', call_id='call_Os5FTdeLEe9J849KDwqnRftW', name='get_employee_detail', type='function_call', id='fc_029ba5f87978f172006a22b2471a00819eb2394fe2409e7327', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"id":2}', call_id='call_02hxmRWs287bpjV60rYdGQAQ', name='get_employee_detail', type='function_call', id='fc_029ba5f87978f172006a22b2471a1081

In [15]:
response = user_input(input_messages)

# Textausgabe der Responses API.
antwort = response.output_text
antwort

'Das durchschnittliche Gehalt beträgt 225.000 (gleiche Einheit wie in den Gehaltsdaten).'